# RQ3, method B: do co-played tracks sound alike? Session coherence and nearest neighbours in sound space

**Research question 3.** *Which tracks are played together, and do they sound alike?*

Method A (frequent itemsets with A-Priori / PCY) answers the first half: which items occur together in sessions more often than chance.
This notebook is **method B**. It answers the question with the **128-dimensional audio embeddings** only, in three parts:

| Part | Question | Measure |
|---|---|---|
| 1 | Do the tracks of one session sound alike? | **session coherence** (mean pairwise cosine), **nearest-neighbour coherence** (mean cosine to the closest other track in the session), share of track pairs in the same pseudo-genre |
| 2 | Is a co-played track a *nearest neighbour* in the whole sound space? | rank of track B among all tracks ordered by similarity to track A; share of pairs where B is in A's nearest 1 % / 5 % |
| 3 | Do the frequent pairs of method A also sound alike? | cosine and same-pseudo-genre share of method A's frequent pairs vs random pairs; Spearman correlation between lift and cosine |

**The key control: a shuffled baseline.** We already know (notebook 03) that the recommended stream covers a narrower region of the
sound space for each user. A session can therefore look coherent just because the *user's taste* is coherent. To separate the
session from the taste, every real session is compared with a **shuffled session**: same user, same stream, same size, but tracks drawn
at random from that user's own real tracks in that stream. *Excess = real - shuffled* measures how much the session itself adds.

**Fairness rules** (as in notebooks 02 and 03): only real listens (played >= `REAL_PCT` %), distinct tracks, the same users in both
streams, the same number of baskets of every size in both streams, an equal number of pairs per stream. All results are associations, not causes.

*Prerequisite:* `python scripts/extract_embeddings_50m.py` has been run (creates `data/processed/emb50m_*.npy`).
Part 3 additionally needs the output file of method A (see section 7); without it, part 3 is skipped.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from scipy import stats
from scipy.sparse import csr_matrix

# ---------------- parameters (change here, then Run All) ----------------
REAL_PCT = 50            # real listen = at least this % played
N_TRACKS = 100           # users need at least this many distinct real tracks in BOTH streams (same users as RQ1)
SESSION_GAP_S = 30 * 60  # new session after a gap longer than this
K_GENRE = 32             # number of pseudo-genres (k-means clusters)
CAP = 15                 # session baskets: max distinct tracks per basket
MAX_BASKETS_PER_USER = 100
MIN_BASKETS = 5          # a user needs at least this many matched baskets in each stream for the per-user comparison
N_REF = 50_000           # part 2: size of the random reference sample of the sound space
N_PAIRS = 50_000         # part 2: co-played pairs per stream
SEED = 42

rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COL = {"organic": "#2a9d8f", "recommended": "#e76f51"}
STREAMS = (("organic", 1), ("recommended", 0))      # name, value of is_organic

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "yambda").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not find data/raw/yambda above " + str(Path.cwd()))
    ROOT = ROOT.parent
DATA, PROC = ROOT / "data" / "raw" / "yambda", ROOT / "data" / "processed"
verdict = []


def add_verdict(part, metric, organic, recommended, higher_means):
    """higher_means: 'broader' or 'narrower' (what a higher value of the metric indicates)."""
    rec_higher = recommended > organic
    broader = rec_higher if higher_means == "broader" else not rec_higher
    verdict.append({"part": part, "metric": metric, "organic": organic, "recommended": recommended,
                    "relative difference": (recommended - organic) / abs(organic) if organic else np.nan,
                    "recommended is": "broader" if broader else "narrower"})


def rank_in_group(g):
    """Position of every element inside its group, keeping the original order within a group."""
    order = np.argsort(g, kind="stable"); gs = g[order]
    start = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    sizes = np.diff(np.r_[start, len(gs)])
    rank = np.empty(len(g), dtype=np.int64)
    rank[order] = np.arange(len(gs)) - np.repeat(start, sizes)
    return rank


def paired(org, rec):
    """Paired comparison over users: medians, bootstrap CI of the median difference, share, Wilcoxon p."""
    d = rec - org
    boot = np.median(d[rng.integers(0, len(d), (1000, len(d)))], axis=1)
    return dict(users=len(d), organic=np.median(org), recommended=np.median(rec), diff=np.median(d),
                lo=np.quantile(boot, 0.025), hi=np.quantile(boot, 0.975),
                share_rec_higher=(d > 0).mean(), p=stats.wilcoxon(d).pvalue)


print("project root:", ROOT)

## 1. Load listens, embeddings, artists and sessions

Same loading code as notebook 03. Every listen gets a user index, its embedding row (`-1` if the track has no embedding, about 3 % of
listens), its artist, and a session id: a new session starts when the user changes or after a gap of more than `SESSION_GAP_S` seconds.

In [ ]:
t0 = time.time()
df = pq.read_table(DATA / "flat/50m/multi_event.parquet",
                   columns=["uid", "item_id", "timestamp", "is_organic", "played_ratio_pct", "event_type"]).to_pandas()
df = df[df["event_type"] == "listen"].drop(columns="event_type").reset_index(drop=True)

emb_ids = np.load(PROC / "emb50m_item_ids.npy")
EN = np.load(PROC / "emb50m_norm.npy")                      # normalised embeddings (unit length), float32
L = int(max(df["item_id"].max(), emb_ids.max())) + 1
item_row = np.full(L, -1, dtype=np.int64); item_row[emb_ids] = np.arange(len(emb_ids))   # item id -> embedding row

art = pd.read_parquet(DATA / "artist_item_mapping.parquet").drop_duplicates("item_id")
item_art = np.full(L, -1, dtype=np.int32); ids = art["item_id"].to_numpy(); ok = ids < L
item_art[ids[ok]] = art["artist_id"].to_numpy()[ok]           # item id -> artist id (-1 = unknown)
del art, ids, ok

item_arr = df["item_id"].to_numpy()
org_arr = df["is_organic"].to_numpy().astype(np.int8)
ratio_arr = df["played_ratio_pct"].to_numpy().astype(np.float32)
erow_arr = item_row[item_arr].astype(np.int32)
ts = df["timestamp"].to_numpy().astype(np.int64)
uid = df["uid"].to_numpy()
new_user = np.empty(len(uid), dtype=bool); new_user[0] = True; new_user[1:] = uid[1:] != uid[:-1]
uidx_arr = (np.cumsum(new_user) - 1).astype(np.int32)
n_users = int(uidx_arr[-1]) + 1
new_sess = np.empty(len(ts), dtype=bool); new_sess[0] = True
new_sess[1:] = new_user[1:] | ((ts[1:] - ts[:-1]) > SESSION_GAP_S)
sid_arr = (np.cumsum(new_sess) - 1).astype(np.int64)
del df, uid, new_user, new_sess

print(f"{len(ts):,} listens, {n_users:,} users, {sid_arr[-1] + 1:,} sessions; embeddings {EN.shape}; "
      f"{(erow_arr >= 0).mean():.1%} of listens have an embedding ({time.time() - t0:.0f} s)")

## 2. Pseudo-genres: spherical k-means (k = 32)

There are no genre labels in Yambda, so "same genre" means *same cluster of the sound space*. We use the same own spherical k-means as
notebook 03 (cosine assignment, renormalised centroids, k-means++ seeding), fitted on a random sample of 150,000 tracks and then used to
assign **every** embedded track. Notebook 03 validates this implementation against scikit-learn; here we only need the labels
(`genre_row`, indexed by embedding row) and the centroids (`C_GENRE`, used in part 3 to give artists a pseudo-genre).

In [ ]:
def spherical_kmeans(X, k, seed, n_iter=60):
    r = np.random.default_rng(seed); n = len(X)
    C = np.empty((k, X.shape[1]), dtype=np.float32); C[0] = X[r.integers(n)]
    dist = 1 - X @ C[0]
    for j in range(1, k):                                   # k-means++ seeding with cosine distance
        pr = np.maximum(dist, 0) ** 2; C[j] = X[r.choice(n, p=pr / pr.sum())]
        dist = np.minimum(dist, 1 - X @ C[j])
    labels = np.full(n, -1)
    for _ in range(n_iter):
        new = np.argmax(X @ C.T, axis=1)
        if (new == labels).mean() > 0.9995:
            labels = new; break
        labels = new
        sums = csr_matrix((np.ones(n, np.float32), (labels, np.arange(n))), shape=(k, n)) @ X
        nrm = np.linalg.norm(sums, axis=1, keepdims=True)
        empty = nrm[:, 0] == 0
        sums[empty] = X[r.integers(n, size=empty.sum())]; nrm[empty] = 1
        C = (sums / np.maximum(nrm, 1e-9)).astype(np.float32)
    return C, labels, float((X * C[labels]).sum(1).mean())    # centroids, labels, mean cosine to own centroid

t1 = time.time()
Xfit = EN[rng.choice(len(EN), min(150_000, len(EN)), replace=False)]
C_GENRE, _, obj = spherical_kmeans(Xfit, K_GENRE, SEED)
genre_row = np.empty(len(EN), dtype=np.int16)                 # pseudo-genre of every embedded track
for s0 in range(0, len(EN), 200_000):
    genre_row[s0:s0 + 200_000] = np.argmax(EN[s0:s0 + 200_000] @ C_GENRE.T, axis=1)
del Xfit
cs = np.bincount(genre_row, minlength=K_GENRE)
print(f"k = {K_GENRE}: mean cosine to centroid {obj:.3f}; cluster sizes min {cs.min():,}, median {np.median(cs):,.0f}, "
      f"max {cs.max():,} tracks ({time.time() - t1:.0f} s)")

## 3. Users and matched session baskets

**Users.** The same users as RQ1 / notebook 03: those with at least `N_TRACKS` distinct real tracks (with an embedding) in *both* streams.

**Baskets.** A basket is the set of distinct real tracks of one user in one session and one stream (organic and recommended plays of
the same session form two separate baskets). We keep the first `CAP` distinct tracks, baskets with at least 2 tracks, at most
`MAX_BASKETS_PER_USER` random baskets per user and stream (so heavy users do not dominate) and finally **the same number of baskets of
every size in both streams**, so basket length cannot explain a difference (larger baskets have lower mean similarity).

Result: `chosen` is a list of `(k, organic basket ids, recommended basket ids)`; basket `i` holds the tracks
`bi_[start[i] : start[i] + size[i]]` of user `b_user[i]`.

In [ ]:
def eligible_users(real_pct, n_tracks):
    """Users with at least n_tracks distinct real, embedded tracks in both streams."""
    m = (ratio_arr >= real_pct) & (erow_arr >= 0)
    k3 = (uidx_arr[m].astype(np.int64) << 33) | (item_arr[m].astype(np.int64) << 1) | org_arr[m].astype(np.int64)
    k3 = np.unique(k3)                                           # distinct (user, track, stream)
    g = (k3 >> 33) * 2 + (k3 & 1)
    cnt = np.bincount(g, minlength=2 * n_users).reshape(n_users, 2)
    return np.flatnonzero((cnt >= n_tracks).all(axis=1))

users = eligible_users(REAL_PCT, N_TRACKS)
elig_u = np.zeros(n_users, dtype=bool); elig_u[users] = True

m = (ratio_arr >= REAL_PCT) & (erow_arr >= 0) & elig_u[uidx_arr]
bs, bo, bi_, bu = sid_arr[m], org_arr[m].astype(np.int64), item_arr[m].astype(np.int64), uidx_arr[m]
_, fi = np.unique(((bs * 2 + bo) << 24) | bi_, return_index=True); fi.sort()   # distinct track per (session, stream)
bs, bo, bi_, bu = bs[fi], bo[fi], bi_[fi], bu[fi]
gb = bs * 2 + bo                                                 # basket id = session x stream
keep = rank_in_group(gb) < CAP
gb, bi_, bu, bo = gb[keep], bi_[keep], bu[keep], bo[keep]
order = np.argsort(gb, kind="stable"); gb, bi_, bu, bo = gb[order], bi_[order], bu[order], bo[order]
start = np.r_[0, np.flatnonzero(gb[1:] != gb[:-1]) + 1]
size = np.diff(np.r_[start, len(gb)]); b_user, b_org = bu[start], bo[start]
del bs, bo, bu, gb, keep, order, fi

cand = np.flatnonzero(size >= 2); cand = cand[rng.permutation(len(cand))]
cand = cand[rank_in_group(b_user[cand].astype(np.int64) * 2 + b_org[cand]) < MAX_BASKETS_PER_USER]
chosen = []
for k in range(2, CAP + 1):
    ik = cand[size[cand] == k]; o = ik[b_org[ik] == 1]; rc = ik[b_org[ik] == 0]
    n_ = min(len(o), len(rc))
    if n_:
        chosen.append((k, rng.choice(o, n_, replace=False), rng.choice(rc, n_, replace=False)))

n_b = sum(len(o) for _, o, _ in chosen)
mean_k = sum(k * len(o) for k, o, _ in chosen) / n_b
print(f"{len(users):,} eligible users; {len(start):,} baskets in total; "
      f"{n_b:,} matched baskets per stream (mean size {mean_k:.2f} tracks)")

## 4. Shuffled baseline: the user's taste without the session

For every user and stream we collect the **pool** of distinct real tracks (with an embedding). `shuffled_rows(groups, k)` draws, for
each basket, `k` *different* tracks at random from the pool of the same user and stream. A shuffled basket therefore has the same user,
stream and size as the real one; only the session structure is removed.

Pools have at least `N_TRACKS` = 100 tracks and baskets at most `CAP` = 15, so the redraw loop for duplicates ends after a few rounds.

In [ ]:
# distinct real tracks per (user, stream), sorted by group = user*2 + stream
m = (ratio_arr >= REAL_PCT) & (erow_arr >= 0) & elig_u[uidx_arr]
key = ((uidx_arr[m].astype(np.int64) * 2 + org_arr[m]) << 32) | item_arr[m].astype(np.int64)
pool = np.unique(key)
pool_grp, pool_item = pool >> 32, pool & 0xFFFFFFFF
g_start = np.searchsorted(pool_grp, np.arange(2 * n_users))
g_size = np.searchsorted(pool_grp, np.arange(2 * n_users), side="right") - g_start
del m, key, pool, pool_grp


def shuffled_rows(groups, k):
    """For each basket group (user*2 + stream), k distinct random tracks of that user in that stream (embedding rows)."""
    st, sz = g_start[groups][:, None], g_size[groups][:, None]
    idx = st + rng.integers(0, sz, (len(groups), k))
    while True:                                  # redraw rows that contain a duplicate
        s = np.sort(idx, 1); bad = (np.diff(s, axis=1) == 0).any(1)
        if not bad.any():
            break
        idx[bad] = st[bad] + rng.integers(0, sz[bad], (bad.sum(), k))
    return item_row[pool_item[idx]]


def real_rows(idxs, k):
    """Embedding rows of the k tracks of each basket in idxs."""
    return item_row[bi_[start[idxs][:, None] + np.arange(k)]]


print(f"pool size per user and stream: median {np.median(g_size[g_size > 0]):,.0f}, min {g_size[g_size > 0].min():,} tracks")

## 5. Part 1: session coherence, real vs shuffled

For every matched basket (real and shuffled) we compute three numbers:

* **coherence** = mean pairwise cosine similarity of its tracks (higher = the whole session stays in one sound),
* **nearest-neighbour coherence** = for each track the cosine to the *closest other track in the same session*, averaged over the tracks.
  High NN-coherence with low coherence means the session consists of several tight groups of similar tracks rather than one spread-out mix,
* **same-genre share** = share of track pairs that fall in the same pseudo-genre.

We average per user and stream (users with at least `MIN_BASKETS` baskets in both streams), then compare the two streams within users:
the **raw** values (these should reproduce notebook 03: coherence about 0.477 organic vs 0.417 recommended) and the **excess over
shuffled** (real - shuffled), which is the effect of the session beyond the user's taste.

In [ ]:
def basket_metrics(rows_, chunk=20_000):
    k = rows_.shape[1]; iu = np.triu_indices(k, 1); out = []
    for s0 in range(0, len(rows_), chunk):
        r = rows_[s0:s0 + chunk]; Ev = EN[r]                         # (n, k, 128)
        G = np.einsum("nkd,njd->nkj", Ev, Ev)
        coh = G[:, iu[0], iu[1]].mean(1)                            # mean pairwise cosine
        G[:, np.arange(k), np.arange(k)] = -np.inf
        nn = G.max(2).mean(1)                                       # mean cosine to the nearest other track in the session
        g = genre_row[r]; same = (g[:, iu[0]] == g[:, iu[1]]).mean(1)   # share of pairs in the same pseudo-genre
        out.append(np.c_[coh, nn, same])
    return np.vstack(out)


t1 = time.time()
METRICS = ["coherence", "nn_coherence", "same_genre"]
recs = []
for k, o, rc in chosen:
    for s, v in STREAMS:
        idxs = o if v == 1 else rc
        shuf = shuffled_rows(b_user[idxs].astype(np.int64) * 2 + v, k)
        for kind, rows_ in (("real", real_rows(idxs, k)), ("shuffled", shuf)):
            M_ = basket_metrics(rows_)
            recs.append(pd.DataFrame({"stream": s, "kind": kind, "size": k, "user": b_user[idxs],
                                      "coherence": M_[:, 0], "nn_coherence": M_[:, 1], "same_genre": M_[:, 2]}))
BK = pd.concat(recs, ignore_index=True)
del recs

# per user: mean over baskets, then excess = real - shuffled
pu = BK.groupby(["user", "stream", "kind"])[METRICS].mean().unstack(["stream", "kind"])
n_bu = BK[BK.kind == "real"].groupby(["user", "stream"]).size().unstack()
pu = pu.loc[n_bu.index[(n_bu >= MIN_BASKETS).all(axis=1)]].copy()
res_b1 = {}
for met in METRICS:
    for s, _ in STREAMS:
        pu[(met + "_excess", s, "")] = pu[(met, s, "real")] - pu[(met, s, "shuffled")]
    res_b1[met + " (real)"] = paired(pu[(met, "organic", "real")].to_numpy(), pu[(met, "recommended", "real")].to_numpy())
    res_b1[met + " (shuffled)"] = paired(pu[(met, "organic", "shuffled")].to_numpy(), pu[(met, "recommended", "shuffled")].to_numpy())
    res_b1[met + " excess over shuffled"] = paired(pu[(met + "_excess", "organic", "")].to_numpy(),
                                                   pu[(met + "_excess", "recommended", "")].to_numpy())
T1 = pd.DataFrame(res_b1).T[["users", "organic", "recommended", "diff", "lo", "hi", "share_rec_higher", "p"]]
print(f"users with >= {MIN_BASKETS} matched baskets in both streams: {len(pu):,}  ({time.time() - t1:.0f} s)\n")
print(T1.to_string(float_format=lambda x: f"{x:.4g}"))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for s, _ in STREAMS:
    for kind, ls in (("real", "-"), ("shuffled", ":")):
        g_ = BK[(BK.stream == s) & (BK.kind == kind)].groupby("size")["coherence"].mean()
        ax[0].plot(g_.index, g_.values, ls, marker="o", ms=3, color=COL[s], label=f"{s}, {kind}")
ax[0].set(title="Session coherence by basket size", xlabel="distinct tracks in the basket", ylabel="mean pairwise cosine")
ax[0].legend(fontsize=8)
bins = np.linspace(min(pu[("coherence_excess", s, "")].min() for s, _ in STREAMS),
                   max(pu[("coherence_excess", s, "")].max() for s, _ in STREAMS), 50)
for s, _ in STREAMS:
    x = pu[("coherence_excess", s, "")]
    ax[1].hist(x, bins=bins, histtype="step", lw=1.6, color=COL[s], label=f"{s} (median {x.median():.3f})")
ax[1].axvline(0, color="k", lw=1)
ax[1].set(title="Per user: coherence excess over shuffled", xlabel="real - shuffled coherence", ylabel="users"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading part 1.** Compare first the *shuffled* rows: they show how coherent the user's taste in each stream is on its own (the
narrower recommended taste from notebook 03 should give a *higher* shuffled coherence in the recommended stream). The *excess* rows then
show which stream builds sessions that hang together **beyond** that taste. A positive excess means the tracks of a session are closer
in sound than random tracks of the same user. The left plot checks that the comparison is not driven by basket size.

## 6. Part 2: is a co-played track a nearest neighbour in the whole sound space?

Part 1 compares tracks *within* a session. Part 2 asks how special a co-played pair is relative to **all** music: from each matched
basket we take one random pair of tracks (A, B) and compute the **rank percentile** of B among all tracks ordered by similarity to A:

`rank pct = share of tracks that are closer to A than B is`  (0 = B is A's nearest neighbour, about 0.5 = B is a random track).

Ranking against all 848k tracks for 100k pairs is too heavy, so we rank against a fixed **random reference sample** of `N_REF` tracks
(an unbiased estimate of the same percentile). We report the median rank percentile and the share of pairs where B is among the
**nearest 1 % / 5 %** of A, for co-played pairs and for pairs from the shuffled baskets, with an equal number of pairs (`N_PAIRS`) per stream.
The **excess** (co-played - shuffled) of the nearest-1 % share has a bootstrap 95 % interval.

In [ ]:
REF = EN[rng.choice(len(EN), min(N_REF, len(EN)), replace=False)]


def one_pair_per_basket(rows_):
    """One random pair of two different tracks from every basket."""
    k = rows_.shape[1]; i = rng.integers(0, k, len(rows_)); j = (i + rng.integers(1, k, len(rows_))) % k
    n = np.arange(len(rows_)); return rows_[n, i], rows_[n, j]


def rank_pct(ra, rb, block=1000):
    """Share of reference tracks closer to a than b is (0 = b is a's nearest neighbour, 0.5 = random)."""
    cab = np.einsum("ij,ij->i", EN[ra], EN[rb]); out = np.empty(len(ra))
    for s0 in range(0, len(ra), block):
        out[s0:s0 + block] = ((EN[ra[s0:s0 + block]] @ REF.T) > cab[s0:s0 + block, None]).mean(1)
    return out


def boot_ci(x, stat=np.mean, n=1000):
    b = np.array([stat(x[rng.integers(0, len(x), len(x))]) for _ in range(n)])
    return np.quantile(b, 0.025), np.quantile(b, 0.975)


t1 = time.time()
n_pick = min(N_PAIRS, n_b)                     # both streams have n_b matched baskets, so the same number of pairs
RP, R2 = {}, []
for s, v in STREAMS:
    A_, B_, Ash, Bsh = [], [], [], []
    for k, o, rc in chosen:
        idxs = o if v == 1 else rc
        a, b = one_pair_per_basket(real_rows(idxs, k)); A_.append(a); B_.append(b)
        a, b = one_pair_per_basket(shuffled_rows(b_user[idxs].astype(np.int64) * 2 + v, k)); Ash.append(a); Bsh.append(b)
    A_, B_, Ash, Bsh = map(np.concatenate, (A_, B_, Ash, Bsh))
    pick = rng.choice(len(A_), n_pick, replace=False)
    for kind, a, b in (("co-played", A_[pick], B_[pick]), ("shuffled", Ash[pick], Bsh[pick])):
        RP[(s, kind)] = rank_pct(a, b)
    for kind in ("co-played", "shuffled"):
        rp = RP[(s, kind)]
        R2.append({"stream": s, "pairs": kind, "n": len(rp), "median rank pct": np.median(rp),
                   "share in nearest 1%": (rp < 0.01).mean(), "share in nearest 5%": (rp < 0.05).mean()})
T2 = pd.DataFrame(R2)
print(T2.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nexcess of co-played over shuffled pairs (share in nearest 1%), 95% bootstrap CI:")
ex1 = {}
for s, _ in STREAMS:
    d = (RP[(s, "co-played")] < 0.01).astype(float) - (RP[(s, "shuffled")] < 0.01).astype(float)
    ex1[s] = d.mean(); lo, hi = boot_ci(d)
    print(f"  {s:12s} {d.mean():+.4f}  [{lo:+.4f}, {hi:+.4f}]")
print(f"({time.time() - t1:.0f} s)")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
x = np.arange(2); w = 0.38
for i, (s, _) in enumerate(STREAMS):
    vals = [T2[(T2.stream == s) & (T2.pairs == p)]["share in nearest 1%"].iloc[0] * 100 for p in ("co-played", "shuffled")]
    ax[0].bar(x + (i - 0.5) * w, vals, w, color=COL[s], label=s)
ax[0].set_xticks(x); ax[0].set_xticklabels(["co-played in a session", "shuffled (same user)"])
ax[0].set(title="Pairs where B is among A's nearest 1% in sound", ylabel="% of pairs"); ax[0].legend()
grid = np.logspace(-4, 0, 200)
for s, _ in STREAMS:
    for kind, ls in (("co-played", "-"), ("shuffled", ":")):
        ax[1].plot(grid, [(RP[(s, kind)] <= g).mean() for g in grid], ls, color=COL[s], label=f"{s}, {kind}")
ax[1].plot(grid, grid, color="#adb5bd", lw=1, label="random track")
ax[1].set(xscale="log", title="How close is B to A in the whole sound space?", xlabel="rank percentile (log)",
          ylabel="share of pairs at or below"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading part 2.** The grey diagonal is what a random track would give. Curves above it mean co-played tracks are unusually close
neighbours in sound. The gap between the solid (co-played) and dotted (shuffled) line of the same colour is what the *session* adds
beyond the user's taste; compare that gap between the two streams.

## 7. Part 3: link to method A, do the frequent pairs sound alike?

This is the comparison the briefing asks for: *do the top co-listened pairs lie in the same pseudo-genre (and close in sound) more often
than random pairs?* It needs the frequent pairs from **method A**, which is implemented by another group member. Until that file
exists, this cell prints a message and skips part 3.

**Interface with method A.** Method A saves its frequent pairs (at its main support threshold) as
`data/processed/rq3_method_a_pairs_50m.parquet` with one row per frequent pair and the columns

| column | meaning |
|---|---|
| `stream` | `"organic"` or `"recommended"` |
| `a`, `b` | the two items of the pair: **artist ids** if `METHOD_A_LEVEL = "artist"` (as notebook 02), **track ids** (`item_id`) if `"track"` |
| `lift` | lift of the pair in that stream |

**What we compute.** Every item gets a sound vector: for tracks the embedding itself, for artists the normalised mean embedding of their
tracks. We take an equal number of frequent pairs per stream and compare them with **random pairs** whose items are drawn in proportion
to how often they occur in that stream's matched baskets (the same independence assumption that lift is based on, so popular items
cannot explain the result). Reported: median cosine, same-pseudo-genre share, their excess over random, and the Spearman correlation
between lift and cosine (do stronger co-listening patterns also sound more alike?).

*Caveat:* frequent pairs share items, so they are not independent; the bootstrap intervals over pairs are somewhat too narrow.

In [ ]:
METHOD_A_LEVEL = "artist"          # "artist" or "track": what a and b in method A's file are
PAIRS_FILE = PROC / "rq3_method_a_pairs_50m.parquet"
T3 = None

if not PAIRS_FILE.exists():
    print(f"Part 3 skipped: {PAIRS_FILE.relative_to(ROOT)} not found. Run method A first (see the table above for the format).")
else:
    # sound vector V[pos[id]] and pseudo-genre G3[pos[id]] for every item id
    if METHOD_A_LEVEL == "artist":
        row_art = item_art[emb_ids].astype(np.int64); okr = row_art >= 0
        A_ids, a_col = np.unique(row_art[okr], return_inverse=True)
        Ssum = csr_matrix((np.ones(okr.sum(), np.float32), (a_col, np.flatnonzero(okr))), shape=(len(A_ids), len(EN))) @ EN
        V = (Ssum / np.maximum(np.linalg.norm(Ssum, axis=1, keepdims=True), 1e-9)).astype(np.float32)
        G3 = np.argmax(V @ C_GENRE.T, axis=1)
        pos = np.full(int(max(A_ids.max(), item_art.max())) + 1, -1, dtype=np.int64); pos[A_ids] = np.arange(len(A_ids))
        to_id = lambda tracks: item_art[tracks]                  # track id -> artist id
        del Ssum, row_art
    else:
        V, G3, pos = EN, genre_row, item_row
        to_id = lambda tracks: tracks

    def lookup(ids):
        ids = np.asarray(ids, dtype=np.int64)
        p = np.full(len(ids), -1, dtype=np.int64); ok_ = (ids >= 0) & (ids < len(pos)); p[ok_] = pos[ids[ok_]]
        return p

    FP = pd.read_parquet(PAIRS_FILE)
    n_eq = int(FP.groupby("stream").size().min())               # equal number of frequent pairs per stream
    rows, link = [], {}
    for s, v in STREAMS:
        P_ = FP[FP.stream == s].sample(n_eq, random_state=SEED)
        pa, pb = lookup(P_["a"]), lookup(P_["b"]); k_ = (pa >= 0) & (pb >= 0)
        cos_fp = np.einsum("ij,ij->i", V[pa[k_]], V[pb[k_]]); same_fp = G3[pa[k_]] == G3[pb[k_]]
        # random baseline: items weighted by how often they occur in this stream's matched baskets
        basket_tracks = np.concatenate([bi_[start[o if v == 1 else rc][:, None] + np.arange(k)].ravel() for k, o, rc in chosen])
        bp = lookup(to_id(basket_tracks)); bp = bp[bp >= 0]
        ra, rb = bp[rng.integers(0, len(bp), 50_000)], bp[rng.integers(0, len(bp), 50_000)]; kk = ra != rb
        cos_rd = np.einsum("ij,ij->i", V[ra[kk]], V[rb[kk]]); same_rd = G3[ra[kk]] == G3[rb[kk]]
        lo, hi = boot_ci(cos_fp, np.median)
        link[s] = dict(cos_fp=cos_fp, lift=P_["lift"].to_numpy()[k_], cos_rd=cos_rd)
        rows.append({"stream": s, "frequent pairs": int(k_.sum()),
                     "median cos: frequent": np.median(cos_fp), "CI lo": lo, "CI hi": hi, "median cos: random": np.median(cos_rd),
                     "cos excess": np.median(cos_fp) - np.median(cos_rd),
                     "same genre: frequent": same_fp.mean(), "same genre: random": same_rd.mean(),
                     "same-genre excess": same_fp.mean() - same_rd.mean(),
                     "Spearman(lift, cos)": stats.spearmanr(link[s]["lift"], cos_fp).statistic})
    T3 = pd.DataFrame(rows).set_index("stream")
    print(f"method A pairs at level '{METHOD_A_LEVEL}', {n_eq:,} per stream")
    print(T3.T.to_string(float_format=lambda x: f"{x:.4f}"))

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    bins = np.linspace(-0.2, 1, 60)
    for s, _ in STREAMS:
        ax[0].hist(link[s]["cos_fp"], bins=bins, histtype="step", lw=1.6, density=True, color=COL[s], label=f"{s}: frequent pairs")
        ax[0].hist(link[s]["cos_rd"], bins=bins, histtype="step", lw=1, ls=":", density=True, color=COL[s], label=f"{s}: random pairs")
        ax[1].scatter(np.log10(link[s]["lift"]), link[s]["cos_fp"], s=4, alpha=0.4, color=COL[s], label=s)
    ax[0].set(title="Sound similarity of method A's frequent pairs", xlabel="cosine", ylabel="density"); ax[0].legend(fontsize=8)
    ax[1].set(title="Lift (method A) vs sound similarity (method B)", xlabel="log10(lift)", ylabel="cosine"); ax[1].legend()
    plt.tight_layout(); plt.show()

## 8. Summary of method B

One row per measure: organic, recommended, and what a higher value in the recommended stream means. Higher coherence, a higher
excess and more nearest-neighbour pairs all mean that co-played tracks stay closer together in sound, i.e. *narrower* in sound.

In [ ]:
for met, label in (("coherence", "session coherence"), ("nn_coherence", "nearest-neighbour coherence"),
                   ("same_genre", "same-pseudo-genre share of track pairs")):
    for suffix in (" (real)", " excess over shuffled"):
        r = res_b1[met + suffix]
        add_verdict("1 sessions", label + suffix, r["organic"], r["recommended"], "narrower")
for col in ("median rank pct", "share in nearest 1%"):
    co = T2[T2.pairs == "co-played"].set_index("stream")[col]
    add_verdict("2 sound neighbours", f"co-played pairs: {col}", co["organic"], co["recommended"],
                "broader" if col == "median rank pct" else "narrower")
add_verdict("2 sound neighbours", "nearest-1% excess over shuffled", ex1["organic"], ex1["recommended"], "narrower")
if T3 is not None:
    for col in ("cos excess", "same-genre excess", "Spearman(lift, cos)"):
        add_verdict("3 link to method A", col, T3.loc["organic", col], T3.loc["recommended", col], "narrower")

V = pd.DataFrame(verdict)
pd.set_option("display.width", 200)
print(V.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print()
print(V.groupby("part")["recommended is"].value_counts().unstack(fill_value=0).to_string())

# small result files for the report (the report notebook / figures only need these)
out = ROOT / "outputs"; out.mkdir(exist_ok=True)
T1.to_csv(out / "rq3_method_b_part1_sessions_50m.csv")
T2.to_csv(out / "rq3_method_b_part2_neighbours_50m.csv", index=False)
if T3 is not None:
    T3.to_csv(out / "rq3_method_b_part3_link_50m.csv")
V.to_csv(out / "rq3_method_b_summary_50m.csv", index=False)
print("\nsaved tables to", out.relative_to(ROOT))

## Conclusion (fill in after running on Yambda-50M)

| Measure | Organic | Recommended | Reading |
|---|---|---|---|
| Session coherence (real) | | | expected about 0.477 vs 0.417 (notebook 03) |
| Coherence of the user's taste (shuffled) | | | |
| Coherence excess over shuffled | | | |
| Nearest-neighbour coherence excess | | | |
| Co-played pairs in the nearest 1 % (excess over shuffled) | | | |
| Method A frequent pairs: cosine excess over random | | | |
| Spearman(lift, cosine) | | | |

**Questions to answer in the text.**
1. Do the tracks of a session sound more alike than the user's taste alone predicts (excess > 0), and in which stream more?
2. Are recommended sessions less coherent because the recommender *mixes* sounds within a session, or because of the user's taste? (Compare the real and shuffled rows.)
3. Do method A and method B agree: are the frequent pairs also sound neighbours, and does a higher lift go with a higher cosine?
   Agreement strengthens the finding; disagreement (co-listened but not similar-sounding) is a result in itself.

**Limits.** Associations only. Pseudo-genres are clusters of a CNN embedding, not real genres. The rank percentile is estimated against
a random sample of `N_REF` tracks. Frequent pairs share items, so their intervals are too narrow. Results are for Yambda-50M only.